# Common JSON Tools

Interactive JSON inspection tool for all files in `dcc/config/schemas`.


In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display


In [7]:

# Locate schema JSON files under dcc/config/schemas (work from any CWD)
def find_schema_dir(start_path: Path):
    start = start_path.resolve()
    for base in [start, *start.parents]:
        candidates = [
            base / 'dcc' / 'config' / 'schemas',
            base / 'config' / 'schemas',
            base / 'schemas',
        ]
        for candidate in candidates:
            if candidate.exists() and candidate.is_dir():
                return candidate
    return None

schema_dir = find_schema_dir(Path.cwd())
if schema_dir is None:
    raise FileNotFoundError(f'No JSON schema files found in dcc/config/schemas from cwd {Path.cwd()} (searched parent paths)')

json_files = sorted(schema_dir.glob('*.json'))
if not json_files:
    raise FileNotFoundError(f'No JSON schema files found in {schema_dir}')

print('Resolved schema directory:', schema_dir)
print('Available JSON schema files:')
for i, fpath in enumerate(json_files, start=1):
    print(f'{i:>2}. {fpath.name}')

selection = input('Enter the index or file name to inspect (default 1): ').strip()
if selection == '':
    selection = '1'

if selection.isdigit():
    idx = int(selection) - 1
    if idx < 0 or idx >= len(json_files):
        raise IndexError('Selection index out of range')
    selected_path = json_files[idx]
else:
    selected_path = schema_dir / selection
    if not selected_path.exists():
        raise FileNotFoundError(f'File not found: {selected_path}')

print(f"\nLoading JSON file: {selected_path}")
with selected_path.open('r', encoding='utf-8') as f:
    json_data = json.load(f)

print('\nTop-level keys:')
if isinstance(json_data, dict):
    print(list(json_data.keys()))
else:
    print(type(json_data))


Resolved schema directory: /workspaces/Engineering-and-Design/dcc/config/schemas
Available JSON schema files:
 1. approval_code_mapping.json
 2. dcc_register_enhanced.json
 3. department_schema.json
 4. discipline_schema.json
 5. document_type_schema.json
 6. master_registry.json

Loading JSON file: /workspaces/Engineering-and-Design/dcc/config/schemas/approval_code_mapping.json

Top-level keys:
['schema_name', 'type', 'mappings', 'description', 'created', 'version']


In [ ]:

# DataFrame helpers for schema inspection
# when print out the table result from JSON file, align the test wrapping with title and make it more readable.


def to_schema_table(data):
    schema_dict = data.get('schema', {}) if isinstance(data, dict) else {}
    rows = [
        {'standard_name': standard_name, 'source_name': source_name}
        for standard_name, source_names in schema_dict.items()
        for source_name in source_names
    ]
    return pd.DataFrame(rows)


def to_parameters_table(data):
    parameters_dict = data.get('parameters', {}) if isinstance(data, dict) else {}
    rows = [{'parameter': k, 'value': v} for k, v in parameters_dict.items()]
    return pd.DataFrame(rows)


def to_row_mapping_table(data):
    row_mapping_dict = data.get('row_mapping', {}) if isinstance(data, dict) else {}
    rows = [
        {'approval_code': approval_code, 'alias': alias}
        for approval_code, aliases in row_mapping_dict.items()
        for alias in aliases
    ]
    return pd.DataFrame(rows)


def to_top_level_table(data):
    if not isinstance(data, dict):
        return pd.DataFrame([{'type': str(type(data)), 'value': data}])

    rows = []
    for key, value in data.items():
        if isinstance(value, dict):
            value_type = 'dict'
            size = len(value)
        elif isinstance(value, list):
            value_type = 'list'
            size = len(value)
        else:
            value_type = type(value).__name__
            size = None
        rows.append({'key': key, 'type': value_type, 'size': size, 'preview': str(value)[:120]})

    return pd.DataFrame(rows)

# Render chosen output tables

top_level_table = to_top_level_table(json_data)
schema_table = to_schema_table(json_data)
parameters_table = to_parameters_table(json_data)
row_mapping_table = to_row_mapping_table(json_data)

print('Resolved schema directory:', schema_dir)
print('Top-Level Summary')
display(top_level_table)

# Additional views for this repo's schema shapes
if isinstance(json_data, dict):
    if 'choices' in json_data:
        print('\nChoices Table')
        display(pd.DataFrame({'choice': json_data['choices']}))

    if 'document_types' in json_data:
        print('\nDocument Types Table')
        display(pd.DataFrame(json_data['document_types']).T)

    if 'tools' in json_data:
        print('\nTools Table')
        display(pd.DataFrame(json_data['tools']).T)

    if 'workflows' in json_data:
        print('\nWorkflows Table')
        display(pd.DataFrame(json_data['workflows']).T)

    # If this file contains approval/document mappings, list each mapping row by row
    if 'mappings' in json_data and isinstance(json_data['mappings'], dict):
        print('\nMapping rows (one per key/value item):')
        mapping_rows = []
        for key, values in json_data['mappings'].items():
            if isinstance(values, list):
                for value in values:
                    mapping_rows.append({'key': key, 'mapped_value': value})
            else:
                mapping_rows.append({'key': key, 'mapped_value': values})
        display(pd.DataFrame(mapping_rows))

print('\nSchema Table')
display(schema_table)

print('Parameters Table')
display(parameters_table)

print('Row Mapping Table')
display(row_mapping_table)


Resolved schema directory: /workspaces/Engineering-and-Design/dcc/config/schemas
Top-Level Summary


,key,type,size,preview
0,schema_name,str,NaN,approval_code_mapping
1,type,str,NaN,status_to_code_mapping
2,mappings,dict,7.0,"{'REJ': ['Rejected', 'REJ'], 'NAP': ['Not Appr..."
3,description,str,NaN,Mapping of approval status variations to stand...
4,created,str,NaN,2026-03-28
5,version,str,NaN,1.0



Mapping rows (one per key/value item):


,key,mapped_value
0,REJ,Rejected
1,REJ,REJ
2,NAP,Not Approved - Revise and resubmit
3,NAP,Not Approved
4,AWC,Approved with Comments
5,AWC,Approved as noted
6,APP,Approved
7,APP,APP
8,INF,For Information
9,INF,INF



Schema Table


""


Parameters Table


""


Row Mapping Table


""
